<a href="https://colab.research.google.com/github/Jenna-learner/23521117_DS200_LAP4/blob/main/LAP4_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bài thực hành DS200 - LAP4: Phân tích bán hàng & tiếp thị với Spark DataFrame
**Fecom Inc. — E-commerce (Berlin, Đức)**

Bộ dữ liệu: 99.441 đơn hàng, 102.727 khách hàng, 3.095 người bán, 32.951 sản phẩm thuộc 72 danh mục, từ 338 thành phố / 28 quốc gia (2022–2024).

Notebook này sử dụng **PySpark DataFrame API** để giải toàn bộ 10 câu (Câu 1–5 bắt buộc + Câu 6–10).

In [ ]:
# Khởi tạo SparkSession
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("Fecom_Sales_Analysis")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.version

'4.0.2'

## Câu 1. Đọc dữ liệu từ các file CSV (tự suy ra kiểu dữ liệu)
Các file dùng dấu phân cách `;`, có dòng tiêu đề. Dùng `inferSchema=True` để Spark tự suy ra kiểu dữ liệu cho mỗi cột.

In [ ]:
# Thư mục chứa dữ liệu (đặt notebook cùng thư mục với các file CSV)
DATA_DIR = "."

def read_csv(name):
    return (spark.read
            .option("header", True)
            .option("sep", ";")
            .option("inferSchema", True)
            .option("encoding", "UTF-8")
            .csv(f"{DATA_DIR}/{name}.csv"))

orders        = read_csv("Orders")
customers     = read_csv("Customer_List")
order_items   = read_csv("Order_Items")
products      = read_csv("Products")
order_reviews = read_csv("Order_Reviews")

for nm, df in [("Orders", orders), ("Customer_List", customers),
               ("Order_Items", order_items), ("Products", products),
               ("Order_Reviews", order_reviews)]:
    print(f"==== {nm}: {df.count():,} dòng x {len(df.columns)} cột ====")
    df.printSchema()

==== Orders: 99,441 dòng x 8 cột ====
root
 |-- Order_ID: string (nullable = true)
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Order_Status: string (nullable = true)
 |-- Order_Purchase_Timestamp: timestamp (nullable = true)
 |-- Order_Approved_At: timestamp (nullable = true)
 |-- Order_Delivered_Carrier_Date: timestamp (nullable = true)
 |-- Order_Delivered_Customer_Date: timestamp (nullable = true)
 |-- Order_Estimated_Delivery_Date: timestamp (nullable = true)

==== Customer_List: 102,727 dòng x 10 cột ====
root
 |-- Customer_Trx_ID: string (nullable = true)
 |-- Subscriber_ID: string (nullable = true)
 |-- Subscribe_Date: date (nullable = true)
 |-- First_Order_Date: date (nullable = true)
 |-- Customer_Postal_Code: string (nullable = true)
 |-- Customer_City: string (nullable = true)
 |-- Customer_Country: string (nullable = true)
 |-- Customer_Country_Code: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Gender: string (nullable = true)

==== Order_Items

## Câu 2. Tổng số đơn hàng, số lượng khách hàng và người bán
- Số đơn hàng: đếm theo `Order_ID` (duy nhất) trong bảng Orders.
- Số khách hàng duy nhất: đếm `Subscriber_ID` riêng biệt trong Customer_List.
- Số người bán: đếm `Seller_ID` riêng biệt trong Order_Items.

In [ ]:
tong_don_hang = orders.select("Order_ID").distinct().count()
# "Khách hàng duy nhất" theo đề = số bản ghi trong Customer_List (102.727)
so_khach_hang = customers.count()
so_subscriber = customers.select("Subscriber_ID").distinct().count()
so_nguoi_ban  = order_items.select("Seller_ID").distinct().count()

print(f"Tổng số đơn hàng           : {tong_don_hang:,}")
print(f"Số khách hàng (Customer_List): {so_khach_hang:,}")
print(f"  - Trong đó subscriber duy nhất: {so_subscriber:,}")
print(f"Số người bán (seller)      : {so_nguoi_ban:,}")

Tổng số đơn hàng           : 99,441
Số khách hàng (Customer_List): 102,727
  - Trong đó subscriber duy nhất: 99,382
Số người bán (seller)      : 3,095


## Câu 3. Số lượng đơn hàng theo quốc gia (giảm dần)
Join `Orders` với `Customer_List` qua `Customer_Trx_ID` để lấy quốc gia của khách, rồi gom nhóm đếm số đơn.

In [ ]:
don_theo_qg = (orders.join(customers, on="Customer_Trx_ID", how="left")
    .groupBy("Customer_Country")
    .agg(F.countDistinct("Order_ID").alias("So_don_hang"))
    .orderBy(F.desc("So_don_hang")))

don_theo_qg.show(30, truncate=False)

+----------------+-----------+
|Customer_Country|So_don_hang|
+----------------+-----------+
|Germany         |41754      |
|France          |12848      |
|Netherlands     |11629      |
|Belgium         |5464       |
|Austria         |5043       |
|Switzerland     |3640       |
|United Kingdom  |3382       |
|Poland          |2139       |
|Czechia         |2034       |
|Italy           |2025       |
|Spain           |1651       |
|Portugal        |1336       |
|Sweden          |975        |
|Denmark         |905        |
|Serbia          |746        |
|Norway          |716        |
|Slovakia        |534        |
|Slovenia        |495        |
|Turkey          |485        |
|Greece          |412        |
|Lithuania       |351        |
|Latvia          |280        |
|Croatia         |254        |
|Estonia         |148        |
|Finland         |81         |
|Luxembourg      |68         |
|Andorra         |46         |
+----------------+-----------+



## Câu 4. Số lượng đơn hàng theo năm, tháng đặt hàng
Năm tăng dần, tháng giảm dần. Lấy năm/tháng từ `Order_Purchase_Timestamp`.

In [ ]:
orders_ts = orders.withColumn(
    "purchase_ts", F.to_timestamp("Order_Purchase_Timestamp"))

don_nam_thang = (orders_ts
    .withColumn("Nam", F.year("purchase_ts"))
    .withColumn("Thang", F.month("purchase_ts"))
    .filter(F.col("Nam").isNotNull())
    .groupBy("Nam", "Thang")
    .agg(F.countDistinct("Order_ID").alias("So_don_hang"))
    .orderBy(F.asc("Nam"), F.desc("Thang")))

don_nam_thang.show(50, truncate=False)

+----+-----+-----------+
|Nam |Thang|So_don_hang|
+----+-----+-----------+
|2022|12   |1          |
|2022|10   |324        |
|2022|9    |4          |
|2023|12   |5673       |
|2023|11   |7544       |
|2023|10   |4631       |
|2023|9    |4285       |
|2023|8    |4331       |
|2023|7    |4026       |
|2023|6    |3245       |
|2023|5    |3700       |
|2023|4    |2404       |
|2023|3    |2682       |
|2023|2    |1780       |
|2023|1    |800        |
|2024|10   |4          |
|2024|9    |16         |
|2024|8    |6512       |
|2024|7    |6292       |
|2024|6    |6167       |
|2024|5    |6873       |
|2024|4    |6939       |
|2024|3    |7211       |
|2024|2    |6728       |
|2024|1    |7269       |
+----+-----+-----------+



## Câu 5. Điểm đánh giá trung bình & số lượng đánh giá theo từng mức
**Xử lý NULL và ngoại lệ:** chỉ giữ `Review_Score` không NULL và nằm trong khoảng hợp lệ 1–5.

In [ ]:
reviews_clean = (order_reviews
    .withColumn("Review_Score", F.col("Review_Score").try_cast("int"))
    .filter(F.col("Review_Score").isNotNull())
    .filter((F.col("Review_Score") >= 1) & (F.col("Review_Score") <= 5)))

tong_so_dg = order_reviews.count()
so_dg_hople = reviews_clean.count()
print(f"Tổng bản ghi đánh giá           : {tong_so_dg:,}")
print(f"Số đánh giá hợp lệ (1-5, có giá): {so_dg_hople:,}")
print(f"Số bị loại (NULL/ngoại lệ)      : {tong_so_dg - so_dg_hople:,}")

diem_tb = reviews_clean.agg(F.round(F.avg("Review_Score"), 3).alias("Diem_TB")).collect()[0]["Diem_TB"]
print(f"\nĐiểm đánh giá trung bình        : {diem_tb}")

print("\nSố lượng đánh giá theo từng mức:")
(reviews_clean.groupBy("Review_Score")
    .agg(F.count("*").alias("So_luong"))
    .orderBy("Review_Score")
    .show())

Tổng bản ghi đánh giá           : 99,270
Số đánh giá hợp lệ (1-5, có giá): 99,223
Số bị loại (NULL/ngoại lệ)      : 47

Điểm đánh giá trung bình        : 4.086

Số lượng đánh giá theo từng mức:
+------------+--------+
|Review_Score|So_luong|
+------------+--------+
|           1|   11424|
|           2|    3151|
|           3|    8179|
|           4|   19141|
|           5|   57328|
+------------+--------+



## Câu 6. Doanh thu năm 2024 theo danh mục sản phẩm
Doanh thu = `Price + Freight_Value`. Lọc các đơn có năm đặt hàng = 2024, join sản phẩm để lấy danh mục.

In [ ]:
orders_2024 = (orders
    .withColumn("Nam", F.year(F.to_timestamp("Order_Purchase_Timestamp")))
    .filter(F.col("Nam") == 2024)
    .select("Order_ID"))

doanh_thu_2024 = (order_items
    .join(orders_2024, on="Order_ID", how="inner")
    .join(products, on="Product_ID", how="left")
    .withColumn("Revenue", F.col("Price") + F.col("Freight_Value"))
    .groupBy("Product_Category_Name")
    .agg(F.round(F.sum("Revenue"), 2).alias("Doanh_thu"),
         F.count("*").alias("So_item"))
    .orderBy(F.desc("Doanh_thu")))

doanh_thu_2024.show(30, truncate=False)

+-------------------------------------+---------+-------+
|Product_Category_Name                |Doanh_thu|So_item|
+-------------------------------------+---------+-------+
|Health_Beauty                        |885191.12|5951   |
|Watches_Gifts                        |771986.75|3703   |
|Bed_Bath_Table                       |650794.7 |5884   |
|Sports_Leisure                       |621999.34|4527   |
|Computers_Accessories                |594771.04|4708   |
|Housewares                           |491576.96|4046   |
|Furniture_Decor                      |476466.13|4118   |
|Auto                                 |404210.57|2619   |
|Baby                                 |299052.56|1776   |
|Cool_Stuff                           |273910.05|1473   |
|Garden_Tools                         |259068.32|1879   |
|Telephony                            |217452.13|2336   |
|Perfumery                            |204562.54|1636   |
|Toys                                 |200634.07|1488   |
|Office_Furnit

## Câu 7. Sản phẩm bán chạy nhất & điểm đánh giá trung bình mỗi sản phẩm
Số lượng bán = số dòng order item theo `Product_ID`. Điểm đánh giá TB mỗi sản phẩm lấy từ Order_Reviews (join qua Order_ID).

In [ ]:
# Số lượng bán ra theo sản phẩm
so_luong_ban = (order_items
    .groupBy("Product_ID")
    .agg(F.count("*").alias("So_luong_ban")))

# Điểm đánh giá TB theo sản phẩm
sp_review = (order_items.select("Order_ID", "Product_ID")
    .join(reviews_clean.select("Order_ID", "Review_Score"), on="Order_ID", how="inner")
    .groupBy("Product_ID")
    .agg(F.round(F.avg("Review_Score"), 2).alias("Diem_TB"),
         F.count("*").alias("So_danh_gia")))

sp_tong_hop = (so_luong_ban
    .join(sp_review, on="Product_ID", how="left")
    .join(products.select("Product_ID", "Product_Category_Name"), on="Product_ID", how="left")
    .orderBy(F.desc("So_luong_ban")))

print("Top 20 sản phẩm bán chạy nhất:")
sp_tong_hop.show(20, truncate=False)

top1 = sp_tong_hop.first()
print(f"\nSản phẩm bán chạy NHẤT: {top1['Product_ID']} "
      f"({top1['Product_Category_Name']}) - {top1['So_luong_ban']:,} lượt bán, "
      f"điểm TB = {top1['Diem_TB']}")

Top 20 sản phẩm bán chạy nhất:
+--------------------------------+------------+-------+-----------+---------------------+
|Product_ID                      |So_luong_ban|Diem_TB|So_danh_gia|Product_Category_Name|
+--------------------------------+------------+-------+-----------+---------------------+
|aca2eb7d00ea1a7b8ebd4e68314663af|527         |4.02   |524        |Furniture_Decor      |
|99a4788cb24856965c36a24e339b6058|488         |3.9    |482        |Bed_Bath_Table       |
|422879e10f46682990de24d770e7f83d|484         |3.95   |486        |Garden_Tools         |
|389d119b48cf3043d311335e499d9c6b|392         |4.12   |391        |Garden_Tools         |
|368c6c730842d78016ad823897a372db|388         |3.92   |388        |Garden_Tools         |
|53759a2ecddad2bb87a079a1f1519f73|373         |3.87   |373        |Garden_Tools         |
|d1c427060a0f73f6b889a5c7c61f2ac4|343         |4.19   |340        |Computers_Accessories|
|53b36df67ebb7c41585e8d54d6772e08|323         |4.19   |320        |Wa

## Câu 8. Hiệu suất giao hàng
Hiệu số (số ngày) = `Order_Delivered_Carrier_Date` − `Shipping_Limit_Date`.
- Giá trị **âm/0**: giao cho đơn vị vận chuyển **đúng hoặc sớm hơn** hạn → tốt.
- Giá trị **dương**: **trễ** hạn.

In [ ]:
delivery = (orders.select("Order_ID",
        F.to_timestamp("Order_Delivered_Carrier_Date").alias("carrier_date"))
    .join(order_items.select("Order_ID",
        F.to_timestamp("Shipping_Limit_Date").alias("limit_date")), on="Order_ID")
    .filter(F.col("carrier_date").isNotNull() & F.col("limit_date").isNotNull())
    .withColumn("Chenh_lech_ngay",
        F.datediff(F.col("carrier_date"), F.col("limit_date"))))

print("Thống kê chênh lệch (ngày) giữa ngày giao carrier và hạn ship:")
delivery.select(
    F.round(F.avg("Chenh_lech_ngay"), 2).alias("TB_ngay"),
    F.min("Chenh_lech_ngay").alias("Min"),
    F.max("Chenh_lech_ngay").alias("Max")).show()

tong = delivery.count()
dung_han = delivery.filter(F.col("Chenh_lech_ngay") <= 0).count()
tre_han  = delivery.filter(F.col("Chenh_lech_ngay") > 0).count()
print(f"Tổng đơn xét       : {tong:,}")
print(f"Đúng/sớm hạn (<=0) : {dung_han:,} ({dung_han/tong*100:.1f}%)")
print(f"Trễ hạn (>0)       : {tre_han:,} ({tre_han/tong*100:.1f}%)")

Thống kê chênh lệch (ngày) giữa ngày giao carrier và hạn ship:
+-------+-----+---+
|TB_ngay|  Min|Max|
+-------+-----+---+
|  -3.44|-1055|118|
+-------+-----+---+

Tổng đơn xét       : 111,456
Đúng/sớm hạn (<=0) : 104,786 (94.0%)
Trễ hạn (>0)       : 6,670 (6.0%)


## Câu 9. Phân nhóm khách hàng (số đơn, giá trị TB đơn, tần suất mua)
Tính cho từng `Subscriber_ID`: số đơn hàng, giá trị trung bình mỗi đơn, tần suất (số đơn / số tháng hoạt động). Sau đó phân nhóm theo số lượng đơn.

In [ ]:
# Giá trị mỗi đơn hàng
order_value = (order_items
    .withColumn("item_value", F.col("Price") + F.col("Freight_Value"))
    .groupBy("Order_ID")
    .agg(F.sum("item_value").alias("Order_Value")))

# Gắn khách hàng + thời điểm mua
cust_orders = (orders
    .withColumn("purchase_ts", F.to_timestamp("Order_Purchase_Timestamp"))
    .join(customers.select("Customer_Trx_ID", "Subscriber_ID"), on="Customer_Trx_ID", how="left")
    .join(order_value, on="Order_ID", how="left"))

cust_metrics = (cust_orders.groupBy("Subscriber_ID").agg(
        F.countDistinct("Order_ID").alias("So_don"),
        F.round(F.avg("Order_Value"), 2).alias("Gia_tri_TB_don"),
        F.round(F.sum("Order_Value"), 2).alias("Tong_chi_tieu"),
        F.min("purchase_ts").alias("lan_dau"),
        F.max("purchase_ts").alias("lan_cuoi"))
    .withColumn("So_thang_hoat_dong",
        F.greatest(F.lit(1), F.months_between("lan_cuoi", "lan_dau") + 1))
    .withColumn("Tan_suat_don_thang",
        F.round(F.col("So_don") / F.col("So_thang_hoat_dong"), 3)))

# Phân nhóm theo số lượng đơn
cust_segment = cust_metrics.withColumn("Nhom_KH",
    F.when(F.col("So_don") == 1, "1. Một lần")
     .when(F.col("So_don") <= 3, "2. Thỉnh thoảng (2-3)")
     .when(F.col("So_don") <= 6, "3. Thường xuyên (4-6)")
     .otherwise("4. Trung thành (7+)"))

print("Phân bố khách hàng theo nhóm:")
(cust_segment.groupBy("Nhom_KH").agg(
    F.count("*").alias("So_khach"),
    F.round(F.avg("Gia_tri_TB_don"), 2).alias("GT_TB_don"),
    F.round(F.avg("Tong_chi_tieu"), 2).alias("Chi_tieu_TB"),
    F.round(F.avg("Tan_suat_don_thang"), 3).alias("Tan_suat_TB"))
 .orderBy("Nhom_KH").show(truncate=False))

print("Ví dụ 10 khách hàng giá trị cao nhất:")
cust_segment.orderBy(F.desc("Tong_chi_tieu")).select(
    "Subscriber_ID","So_don","Gia_tri_TB_don","Tong_chi_tieu","Tan_suat_don_thang","Nhom_KH"
).show(10, truncate=False)

Phân bố khách hàng theo nhóm:
+---------------------+--------+---------+-----------+-----------+
|Nhom_KH              |So_khach|GT_TB_don|Chi_tieu_TB|Tan_suat_TB|
+---------------------+--------+---------+-----------+-----------+
|1. Một lần           |93099   |161.43   |161.43     |1.0        |
|2. Thỉnh thoảng (2-3)|2948    |148.65   |300.63     |1.122      |
|3. Thường xuyên (4-6)|44      |176.55   |756.3      |1.181      |
|4. Trung thành (7+)  |5       |118.49   |983.05     |1.14       |
+---------------------+--------+---------+-----------+-----------+

Ví dụ 10 khách hàng giá trị cao nhất:
+--------------------------------+------+--------------+-------------+------------------+---------------------+
|Subscriber_ID                   |So_don|Gia_tri_TB_don|Tong_chi_tieu|Tan_suat_don_thang|Nhom_KH              |
+--------------------------------+------+--------------+-------------+------------------+---------------------+
|0a0a92112bd4c708ca5fde585afaa872|1     |13664.08      |136

## Câu 10. Xếp hạng seller theo tổng doanh thu & số đơn bán được
Tính cho mỗi `Seller_ID`: tổng doanh thu (`Price + Freight_Value`) và số đơn hàng riêng biệt; xếp hạng theo doanh thu giảm dần.

In [ ]:
seller_stats = (order_items
    .withColumn("Revenue", F.col("Price") + F.col("Freight_Value"))
    .groupBy("Seller_ID")
    .agg(F.round(F.sum("Revenue"), 2).alias("Tong_doanh_thu"),
         F.countDistinct("Order_ID").alias("So_don"),
         F.count("*").alias("So_item")))

w = Window.orderBy(F.desc("Tong_doanh_thu"))
seller_rank = seller_stats.withColumn("Hang", F.row_number().over(w))

print("Top 20 seller theo doanh thu:")
seller_rank.select("Hang","Seller_ID","Tong_doanh_thu","So_don","So_item").show(20, truncate=False)

Top 20 seller theo doanh thu:
+----+--------------------------------+--------------+------+-------+
|Hang|Seller_ID                       |Tong_doanh_thu|So_don|So_item|
+----+--------------------------------+--------------+------+-------+
|1   |4869f7a5dfa277a7dca6462dcf3b52b2|249640.7      |1132  |1156   |
|2   |7c67e1448b00f6e969d365cea6b010ab|239536.44     |982   |1364   |
|3   |53243585a1d6dc2643021fd1853d8905|235856.68     |358   |410    |
|4   |4a3ca9315b744ce9f8e9374361493884|235539.96     |1806  |1987   |
|5   |fa1c13f2614d7b5c4749cbc52fecda94|204084.73     |585   |586    |
|6   |da8622b14eb17ae2831f4ac5b9dab84a|185192.32     |1314  |1551   |
|7   |7e93a43ef30c4f03f38b393420bc753a|182754.05     |336   |340    |
|8   |1025f0e2d44d7041d6cf58b6550e0bfa|172860.69     |915   |1428   |
|9   |7a67c85e85bb2ce8582c35f2203ad736|162648.38     |1160  |1171   |
|10  |955fee9216a65b617aa5c0531780ce60|160602.68     |1287  |1499   |
|11  |6560211a19b47992c3666cc44a7e94c0|151265.77     |1854  

In [ ]:
spark.stop()